# 02 Model: from the accuracy trap to a review queue

**Step 4:** split by time and show the accuracy trap.
**Step 5:** weighted logistic regression, isolation forest and gradient boosting, compared on a validation period.
**Step 6:** final evaluation on the test period, with a threshold set by the team's review capacity.

In [1]:
import matplotlib.pyplot as plt

# One consistent, colour-blind-safe look for every chart in this project
BLUE, ORANGE, AQUA, YELLOW, GREY = "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#8a8984"
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 150, "savefig.bbox": "tight",
    "figure.facecolor": "white", "axes.facecolor": "white",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": GREY, "axes.grid": True, "grid.color": "#e6e5e1", "grid.linewidth": 0.8,
    "axes.axisbelow": True, "axes.titleweight": "bold", "axes.titlesize": 12,
    "axes.titlelocation": "left", "axes.labelcolor": "#52514e",
    "xtick.color": "#52514e", "ytick.color": "#52514e", "font.size": 10,
    "legend.frameon": False, "lines.linewidth": 2,
    "axes.prop_cycle": plt.cycler(color=[BLUE, ORANGE, AQUA, YELLOW]),
})

import numpy as np
import pandas as pd

df = pd.read_parquet("../data/processed/transactions_model.parquet")
FEATURES = ["hour", "is_transfer", "log_amount", "log_balance_before", "empties_account",
            "log_receiver_balance_before", "receiver_balance_zero"]

## 1. Split by time, not at random

A fraud model is always used on **future** transactions, so it must be tested that way. A random split would let the model learn from transactions that happened after the ones it is tested on, including the same fraudsters' later moves.

The 14 normal days are split in order:

- **Train:** the first 8 days. The models learn here.
- **Validation:** the next 2 days. Models are compared and chosen here.
- **Test:** the last 4 days. Used once, at the end, for the numbers reported to the business.

Choosing a model on the test set would quietly turn it into a second validation set, and the final scores would be optimistic.

In [2]:
days = sorted(df.day.unique())
train_days, val_days, test_days = days[:8], days[8:10], days[10:]
train, val, test = (df[df.day.isin(d)] for d in (train_days, val_days, test_days))
for name, part, d in [("Train", train, train_days), ("Validation", val, val_days), ("Test", test, test_days)]:
    print(f"{name:11s} days {d[0]:2d}-{d[-1]:2d}: {len(part):>9,} transactions, {part.is_fraud.sum():>5,} fraud")
X_train, y_train = train[FEATURES], train.is_fraud
X_val, y_val = val[FEATURES], val.is_fraud
X_test, y_test = test[FEATURES], test.is_fraud

Train       days  0-10: 1,568,556 transactions, 2,157 fraud
Validation  days 11-12:   337,997 transactions,   540 fraud
Test        days 13-16:   697,399 transactions, 1,068 fraud


## 2. Baseline 1: predict 'not fraud' for everything

In [3]:
from sklearn.metrics import accuracy_score, recall_score

none = np.zeros(len(y_test), dtype=int)
print(f"Accuracy {accuracy_score(y_test, none):.2%}  |  recall {recall_score(y_test, none):.0%}  |  "
      f"fraud let through: {test.loc[test.is_fraud == 1, 'amount'].sum() / 1e6:,.0f} million")

Accuracy 99.85%  |  recall 0%  |  fraud let through: 1,636 million


**99.85% accurate, and every single fraud gets through.** This is the accuracy trap, and it is why a fraud model is never judged on accuracy.

## 3. Baseline 2: the provider's existing rule

In [4]:
raw = pd.read_parquet("../data/raw/paysim.parquet", columns=["step", "type", "isFlaggedFraud", "isFraud"])
raw = raw[raw.type.isin(["TRANSFER", "CASH_OUT"]) & ((raw.step - 1) // 24).isin(test_days)]
print(f"Existing rule on the test days: flagged {raw.isFlaggedFraud.sum()}, "
      f"recall {recall_score(raw.isFraud, raw.isFlaggedFraud):.1%}")

Existing rule on the test days: flagged 1, recall 0.1%


The existing rule targets only giant transfers. Over the four test days it catches almost nothing. Any model has to beat both baselines on **recall at a workable number of flags**.